# v4 — reanalise no otimo, desconfundimento e kernel projetado

Esta versao existe para consertar quatro problemas do desenho anterior. Dois deles podem
invalidar afirmacoes que ja estavam sendo escritas.

**(1) O TOST anterior comparava os mapas no lugar errado.** O `delta` era a media sobre as 12
bandwidths dentro de cada fold — incluindo `c=2.0`, onde os dois mapas ja estao arruinados. Como
o ZZ degrada mais rapido fora do otimo (concentra mais), a media penaliza o ZZ por regioes que
ninguem usaria. A afirmacao "o emaranhamento piora" pode ser artefato do agregado. Aqui cada
mapa e' avaliado **no seu proprio otimo de bandwidth**, escolhido sem olhar o fold de teste.

**(2) Qubits e features estavam confundidos.** `SelectKBest(k=N_QUBITS)` fazia o modelo de 4
qubits ver 4 features e o de 10 ver 10. "A desvantagem cresce com qubits" pode ser "cresce com
dimensionalidade". O braco `fixo` mantem `K_FIX` features para todos os qubits, reencodando
ciclicamente (data re-uploading), de modo que so a largura do circuito varia.

**(3) Faltava o kernel projetado.** O kernel de fidelidade com mapa ZZ e' justamente a familia
que a teoria ja preve concentrar (Thanasilp et al.). Huang et al. propuseram os *projected
quantum kernels* para escapar disso. Sem testa-los, a conclusao nao fala sobre kernels
quanticos — fala sobre a familia que a literatura ja considera problematica.

**(4) Escopo declarado.** Mesmo com tudo isso, o alcance e': mapa ZZ, `reps=2`, dois tipos de
kernel, um corpus. Isso vai para as limitacoes, nao para o abstract.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time, os, hashlib
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

In [ ]:
PKL_PATH = "features_v5.pkl"
META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]

tam = os.path.getsize(PKL_PATH)
assert tam == 11535935, f"pickle truncado? {tam} bytes (esperado 11535935)"
feat = pd.read_pickle(PKL_PATH)
FEATS = [col for col in feat.columns if col not in META]
X_all = feat[FEATS].values.astype(float)
y_all = feat["label"].values.astype(int)
groups_all = feat["speaker_id"].values
painel("dados", [f"{X_all.shape[0]} gravacoes &middot; {len(FEATS)} features &middot; "
                 f"{pd.unique(groups_all).size} falantes",
                 f"prevalencia = {y_all.mean():.3f}", f"pickle integro ({tam} bytes)"])

## 2. Maquinaria

Alem do kernel de fidelidade, o **kernel projetado**: em vez de comparar estados inteiros,
extrai o vetor de Bloch de cada qubit (as matrizes de densidade reduzidas de 1 corpo) e aplica
uma gaussiana sobre esses `3q` numeros. Foi proposto exatamente para evitar a concentracao que
mata o kernel de fidelidade.

Verificacao embutida: sem emaranhamento os vetores de Bloch tem norma 1 (estado puro por
qubit); com emaranhamento a norma cai. Se essa checagem falhar, ha erro na extracao.

In [ ]:
def _state_fn(n_qubits, reps, entangle):
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x):
        for _ in range(reps):
            for i in range(n_qubits):
                qml.Hadamard(wires=i); qml.RZ(2.0 * x[..., i], wires=i)
            if entangle:
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                    qml.RZ(2.0 * (np.pi - x[..., i]) * (np.pi - x[..., i + 1]), wires=i + 1)
                    qml.CNOT(wires=[i, i + 1])
        return qml.state()
    return circuit

def states(X, q, reps=2, ent=True, batch=512):
    f = _state_fn(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s + batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B):
    return np.abs(A @ B.conj().T) ** 2

def bloch(S, q):
    """Vetores de Bloch por qubit a partir do statevector -> (n, 3q)."""
    n = S.shape[0]
    psi = S.reshape((n,) + (2,) * q)
    out = np.empty((n, 3 * q))
    for k in range(q):
        m = np.moveaxis(psi, 1 + k, 1).reshape(n, 2, -1)
        rho = np.einsum('nik,njk->nij', m, m.conj())
        out[:, 3*k+0] = 2 * rho[:, 0, 1].real
        out[:, 3*k+1] = -2 * rho[:, 0, 1].imag
        out[:, 3*k+2] = (rho[:, 0, 0] - rho[:, 1, 1]).real
    return out

def gram_proj(Ba, Bb, gamma):
    return np.exp(-gamma * ((Ba[:, None, :] - Bb[None, :, :]) ** 2).sum(-1))

def mediana_gamma(Ba):
    D2 = ((Ba[:, None, :] - Ba[None, :, :]) ** 2).sum(-1)
    m = np.median(D2[np.triu_indices_from(D2, k=1)])
    return 1.0 / m if m > 0 else 1.0

def cycle_encode(A, q):
    """Repete as k features ciclicamente ate preencher q qubits (data re-uploading)."""
    k = A.shape[1]
    return A[:, [j % k for j in range(q)]]

def fit_surrogate(K_q, Xa):
    D2 = ((Xa[:, None, :] - Xa[None, :, :]) ** 2).sum(-1)
    loss = lambda lg: np.linalg.norm(K_q - np.exp(-np.exp(lg) * D2))
    r = minimize_scalar(loss, bounds=(-8, 6), method="bounded")
    return float(np.exp(r.x)), float(r.fun) / np.sqrt(K_q.size)

def off_diag_sd(K):
    return float(K[np.triu_indices_from(K, k=1)].std())

def subsample(ytr, gtr, n_per_class, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= n_per_class:
                break
        idx += take[:n_per_class]
    return np.array(idx)

# checagem do kernel projetado
_t = np.random.default_rng(0).uniform(0, np.pi, (30, 6))
_n1 = np.linalg.norm(bloch(states(_t, 6, ent=False), 6).reshape(30, 6, 3), axis=2).mean()
_n2 = np.linalg.norm(bloch(states(_t, 6, ent=True), 6).reshape(30, 6, 3), axis=2).mean()
assert _n1 > 0.999 and _n2 < 0.95, "extracao de Bloch inconsistente"
painel("verificacao do kernel projetado",
       [f"norma de Bloch sem emaranhamento = {_n1:.4f} (esperado 1)",
        f"norma de Bloch com emaranhamento = {_n2:.4f} (esperado &lt; 1)"],
       destaque="A queda da norma <b>e'</b> o emaranhamento: e' exatamente a informacao que o "
                "kernel projetado descarta e o de fidelidade retem.")

## 3. Bracos e orcamento

Tres bracos, cada um respondendo a uma das criticas:

| braco | kernel | features | responde a |
|---|---|---|---|
| `fid-acoplado` | fidelidade | `k = q` (como antes) | reproduz a v3, serve de referencia |
| `fid-fixo` | fidelidade | `k = K_FIX`, reencodado | desconfunde qubits de dimensionalidade |
| `proj-acoplado` | projetado | `k = q` | testa a familia que escapa da concentracao |

Cada braco roda com os dois mapas (`zz` e `prod`). O custo e' 3x o da v3 — o painel abaixo
mostra o numero exato antes de comecar. Nada e' reduzido automaticamente: se quiser encurtar,
`N_REPS` e' a alavanca de menor perda (repeticoes reamostram dentro dos mesmos folds e nao
adicionam informacao independente); `N_FOLDS` e' a que **nao** convem mexer, porque e' a unica
que sustenta os intervalos de confianca.

In [ ]:
QUBITS = [4, 6, 8, 10]
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)
MAPAS = [("zz", True), ("prod", False)]
BRACOS = [("fid-acoplado", "fid", "acoplado"),
          ("fid-fixo",     "fid", "fixo"),
          ("proj-acoplado", "proj", "acoplado")]
K_FIX = 4
N_PER_CLASS, N_FOLDS, N_REPS, C_SVM, REPS_CIRCUITO = 200, 10, 3, 1.0, 2
MARGEM_EQUIV = 0.01
CSV_BRUTO = "sweep_v4_bruto.csv"

n_pontos = len(BRACOS) * len(MAPAS) * len(QUBITS) * len(SCALES) * N_FOLDS * N_REPS
painel("orcamento", [
    f"{len(BRACOS)} bracos &times; {len(MAPAS)} mapas &times; {len(QUBITS)} qubits &times; "
    f"{len(SCALES)} bandwidths &times; {N_FOLDS} folds &times; {N_REPS} reps",
    f"= <b>{n_pontos}</b> avaliacoes de kernel",
    f"estimativa: {n_pontos*0.95/60:.0f}-{n_pontos*1.6/60:.0f} min em CPU",
    f"resultados brutos serao salvos em <b>{CSV_BRUTO}</b>"],
    destaque="Salvar o bruto permite refazer toda a analise das secoes 5 e 6 sem rodar a grade "
             "de novo. Se a sessao cair, recarregue o CSV e siga.")

In [ ]:
rows = []
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=7)
t0 = time.time()

for fold, (tr, te) in enumerate(sgkf.split(X_all, y_all, groups_all)):
    for rep in range(N_REPS):
        rng = np.random.default_rng(1000 * fold + rep)
        sub = subsample(y_all[tr], groups_all[tr], N_PER_CLASS, rng)
        ytr, yte = y_all[tr][sub], y_all[te]
        imp = SimpleImputer(strategy="median").fit(X_all[tr][sub])
        Xtr_i, Xte_i = imp.transform(X_all[tr][sub]), imp.transform(X_all[te])

        for braco, kern, modo in BRACOS:
            k_sel = None
            for q in QUBITS:
                k = q if modo == "acoplado" else K_FIX
                if k != k_sel:
                    sel = SelectKBest(f_classif, k=k).fit(Xtr_i, ytr)
                    sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr_i))
                    A0 = sc.transform(sel.transform(Xtr_i))
                    B0 = np.clip(sc.transform(sel.transform(Xte_i)), 0, np.pi)
                    k_sel = k
                A = A0 if modo == "acoplado" else cycle_encode(A0, q)
                B = B0 if modo == "acoplado" else cycle_encode(B0, q)

                for cval in SCALES:
                    Ac, Bc = A * cval, B * cval
                    for nome, ent in MAPAS:
                        S1, S2 = states(Ac, q, REPS_CIRCUITO, ent), states(Bc, q, REPS_CIRCUITO, ent)
                        if kern == "fid":
                            K1, K2 = gram_fid(S1, S1), gram_fid(S2, S1)
                        else:
                            P1, P2 = bloch(S1, q), bloch(S2, q)
                            gp = mediana_gamma(P1)          # heuristica da mediana, so no treino
                            K1, K2 = gram_proj(P1, P1, gp), gram_proj(P2, P1, gp)
                        m = SVC(kernel="precomputed", C=C_SVM).fit(K1, ytr)
                        auc_q = roc_auc_score(yte, m.decision_function(K2))
                        g_s, resid = fit_surrogate(K1, Ac)
                        ms = SVC(kernel="rbf", gamma=g_s, C=C_SVM).fit(Ac, ytr)
                        rows.append(dict(braco=braco, mapa=nome, fold=fold, rep=rep,
                                         n_qubits=q, n_feat=k, c=cval, auc_q=auc_q,
                                         auc_surr=roc_auc_score(yte, ms.decision_function(Bc)),
                                         resid=resid, gram_off_sd=off_diag_sd(K1)))
    print(f"fold {fold} ok — {time.time() - t0:.0f}s")

df = pd.DataFrame(rows)
df.to_csv(CSV_BRUTO, index=False)
painel("grade concluida",
       [f"{len(df)} linhas em {time.time()-t0:.0f}s", f"salvo em {CSV_BRUTO}"],
       tabela=df.groupby("braco").auc_q.agg(["max", "mean"]).round(4).reset_index())

## 4. Correcao (1) — cada mapa no seu proprio otimo

A bandwidth de cada mapa e' escolhida por **leave-one-fold-out**: para avaliar o fold *i*, o
`c` otimo vem da media dos *outros* folds. Nenhuma informacao do fold de teste entra na escolha,
e nao e' preciso rodar a grade de novo.

Comparar no otimo e' a unica comparacao com sentido pratico: ninguem usaria um kernel com a
bandwidth errada de proposito.

In [ ]:
def delta_no_otimo(d, braco):
    g = d[d.braco == braco]
    linhas = []
    for q in sorted(g.n_qubits.unique()):
        gq = g[g.n_qubits == q]
        por_fold = gq.groupby(["mapa", "fold", "c"]).auc_q.mean().reset_index()
        deltas = []
        for f in sorted(por_fold.fold.unique()):
            escolhido = {}
            for mp in ("zz", "prod"):
                fora = por_fold[(por_fold.mapa == mp) & (por_fold.fold != f)]
                c_star = fora.groupby("c").auc_q.mean().idxmax()       # so folds de fora
                dentro = por_fold[(por_fold.mapa == mp) & (por_fold.fold == f) &
                                  (por_fold.c == c_star)]
                escolhido[mp] = (float(dentro.auc_q.iloc[0]), c_star)
            deltas.append(escolhido["zz"][0] - escolhido["prod"][0])
        d_ = np.array(deltas); n = len(d_)
        se = d_.std(ddof=1) / np.sqrt(n); tc = tdist.ppf(0.975, n - 1)
        lo, hi = d_.mean() - tc * se, d_.mean() + tc * se
        if lo > -MARGEM_EQUIV and hi < MARGEM_EQUIV:      ver = "equivalente"
        elif hi < -MARGEM_EQUIV or lo > MARGEM_EQUIV:     ver = "diferenca real"
        else:                                             ver = "inconclusivo"
        linhas.append(dict(qubits=q, folds=n, delta=round(d_.mean(), 4),
                           ic95_lo=round(lo, 4), ic95_hi=round(hi, 4), veredito=ver))
    return pd.DataFrame(linhas)

ot = delta_no_otimo(df, "fid-acoplado")

# comparacao com o metodo antigo (media sobre toda a grade)
g = df[df.braco == "fid-acoplado"]
antigo = (g.pivot_table(index=["n_qubits", "c", "fold"], columns="mapa", values="auc_q")
          .assign(d=lambda z: z["zz"] - z["prod"]).reset_index()
          .groupby(["n_qubits", "fold"]).d.mean().groupby("n_qubits").mean().round(4))
ot["delta_metodo_antigo"] = ot.qubits.map(antigo)

if (ot.veredito == "diferenca real").any():
    dest = ("O achado do emaranhamento <b>sobrevive</b> a comparacao no otimo. "
            "Isso e' o que a teoria de concentracao nao prediz trivialmente — e' o resultado "
            "mais original do artigo.")
elif (ot.veredito == "equivalente").all():
    dest = ("No otimo os dois mapas sao <b>equivalentes</b>. A desvantagem do ZZ era artefato "
            "da media sobre bandwidths ruins. A afirmacao 'o emaranhamento piora' <b>cai</b> "
            "e precisa sair do texto.")
else:
    dest = ("Inconclusivo no otimo. Nao afirme nem vantagem nem equivalencia — reporte o IC.")

painel("delta ZZ - prod, cada mapa no seu otimo (LOFO)",
       [f"margem: &plusmn;{MARGEM_EQUIV} de AUC", f"unidade: fold (n={N_FOLDS})",
        "coluna final = o que o metodo antigo (media sobre a grade) dava"],
       tabela=ot, destaque=dest)

## 5. Correcao (2) — qubits nao sao features

Se a desvantagem do ZZ escalar com qubits no braco `acoplado` mas **nao** no `fixo`, o que
escalava era dimensionalidade, nao largura do circuito. As duas leituras levam a conclusoes
diferentes no artigo.

In [ ]:
cmp = []
for braco in ("fid-acoplado", "fid-fixo"):
    t = delta_no_otimo(df, braco); t["braco"] = braco; cmp.append(t)
cmp = pd.concat(cmp)
tab = cmp.pivot(index="qubits", columns="braco", values="delta").reset_index()

r_ac, p_ac = spearmanr(cmp[cmp.braco == "fid-acoplado"].qubits,
                       cmp[cmp.braco == "fid-acoplado"].delta)
r_fx, p_fx = spearmanr(cmp[cmp.braco == "fid-fixo"].qubits,
                       cmp[cmp.braco == "fid-fixo"].delta)
if r_ac < -0.5 and r_fx > -0.5:
    dest = ("A escala com qubits aparece so no braco acoplado: o que crescia era a "
            "<b>dimensionalidade</b>, nao a largura do circuito. Corrija a afirmacao.")
elif r_ac < -0.5 and r_fx < -0.5:
    dest = ("A escala persiste com features fixas: e' <b>largura de circuito</b> mesmo. "
            "A afirmacao original se sustenta e fica mais forte.")
else:
    dest = "Sem tendencia monotona clara em nenhum dos bracos. Nao afirme escala com qubits."

painel("delta no otimo: features acopladas vs fixas",
       [f"braco fixo usa K_FIX={K_FIX} features reencodadas em todos os qubits",
        f"Spearman(qubits, delta) acoplado = {r_ac:+.3f} (p={p_ac:.2g})",
        f"Spearman(qubits, delta) fixo &nbsp;&nbsp;&nbsp;= {r_fx:+.3f} (p={p_fx:.2g})"],
       tabela=tab, destaque=dest)

## 6. Correcao (3) — o kernel projetado escapa?

Se o kernel projetado tambem acompanhar o surrogate classico, a auditoria deixa de falar apenas
sobre a familia que a teoria ja condenava. Se ele **nao** acompanhar, o artigo tem que dizer
isso — e a conclusao passa a ser "kernels de fidelidade falham; projetados merecem investigacao",
que e' mais util e mais honesta que uma condenacao geral.

In [ ]:
lin, tb = [], []
for braco in ("fid-acoplado", "proj-acoplado"):
    g = df[df.braco == braco]
    a = g.groupby(["n_qubits", "c"])[["auc_q", "auc_surr", "gram_off_sd"]].mean().reset_index()
    r, p = spearmanr(a.auc_q, a.auc_surr)
    dd = (a.auc_q - a.auc_surr).abs()
    lin.append(f"<b>{braco}</b>: Spearman(auc_q, auc_surr) = {r:+.3f} (p={p:.2g}) &nbsp;|&nbsp; "
               f"|dif| mediana={dd.median():.4f} max={dd.max():.4f}")
    tb.append(dict(braco=braco, auc_max=round(a.auc_q.max(), 4),
                   rho_surrogate=round(r, 3), dif_mediana=round(dd.median(), 4),
                   concentracao_min=round(a.gram_off_sd.min(), 5)))
tb = pd.DataFrame(tb)

r_proj = tb[tb.braco == "proj-acoplado"].rho_surrogate.iloc[0]
if r_proj > 0.9:
    dest = ("O kernel projetado <b>tambem</b> acompanha o surrogate classico. A auditoria "
            "cobre as duas familias — conclusao bem mais ampla e defensavel.")
else:
    dest = ("O kernel projetado <b>nao</b> acompanha o surrogate. A conclusao do artigo tem que "
            "se restringir a kernels de fidelidade, e os projetados viram trabalho futuro "
            "(ou uma secao inteira). Nao generalize.")

painel("fidelidade vs projetado", lin, tabela=tb, destaque=dest)

## 7. O que reportar

- Se a secao 4 derrubar o achado do emaranhamento, **isso vai para o texto**, nao para a gaveta:
  a diferenca entre a analise agregada e a analise no otimo e' ela propria um resultado
  metodologico, do mesmo tipo que a pseudorreplicacao dos subgrupos.
- Escopo real do artigo: mapa ZZ, `reps=2`, kernels de fidelidade e projetado, um corpus.
- A auditoria nao precisa de hardware: nao ha alegacao de vantagem quantica a validar, e a
  simulacao de statevector e' exata.